<a href="https://colab.research.google.com/github/D2718281828nis/BioMedAI-sEEG-core-of-epilepsy/blob/main/sEEG_extreme_event_detector_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/D2718281828nis/BioMedAI-sEEG-core-of-epilepsy/blob/main/sEEG_extreme_event_detector_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data-driven extreme-event detection in all sEEG channels

This Google Colab notebook scans **every time series except channels whose names begin with `MKR`**, finds unusually energetic/transient windows without assuming an event time or channel, reports event intervals, and visualizes the strongest event.

The ensemble combines conventional time-domain features with Dynamic Time Warping, Detrended Fluctuation Analysis, Discrete Wavelet Transform energies, and Kuramoto phase synchronization in delta, theta, alpha, beta, and gamma rhythms. Each method is robustly standardized against this recording before the strongest method responses are combined.

**Section 4b adds one more thing, still without any apriori time:** the EDF’s own EDF+ annotation channel is decoded (cp1251) and searched for a seizure-labelled marker. When found, that time is drawn on every figure below as a distinct **EDF-annotated peak** line, and Section 10 quantifies how each individual method—and the combined ensemble—would have scored *that* moment, separately from whatever window the blind ensemble ranks highest. The two need not agree, and on `sEEG-HFOs-8.edf` they do not: dense interictal activity can outscore the true seizure in a purely statistical scan.

> **Important:** “extreme” means statistically unusual within this recording, not necessarily epileptic. Movement, electrode artifacts, saturation, disconnected contacts, and reference noise can score highly. Review raw signals and acquisition metadata; this exploratory notebook is not a clinical device.

## 1. Install and import dependencies

In [ ]:
%pip install -q "mne>=1.6" "edfio>=0.4" "numpy>=1.23" "pandas>=1.5" "matplotlib>=3.7" "seaborn>=0.12" "scipy>=1.10" "PyWavelets>=1.5" "numba>=0.58"

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
import pywt
import seaborn as sns
from IPython.display import display
from numba import njit, prange
from scipy.signal import butter, hilbert, sosfiltfilt

mne.set_log_level("WARNING")
plt.rcParams.update({"figure.figsize": (16, 6), "axes.spines.top": False,
                     "axes.spines.right": False})

## 2. Configuration

- Windows overlap so a transient near a boundary is not missed.
- `CHANNEL_FRACTION` controls how many of the most abnormal channels contribute to each recording-wide score. A small fraction detects focal events; increase it for spatially widespread events.
- `THRESHOLD_MAD` is the automatic threshold in robust standard-deviation units above the score median.
- Adjacent flagged windows are merged into one event interval.

In [ ]:
EDF_NAME = "sEEG-HFOs-8.edf"
WINDOW_SECONDS = 4.0       # long enough to represent slow delta activity
STEP_SECONDS = 0.50
CHANNEL_FRACTION = 0.10
THRESHOLD_MAD = 6.0
MERGE_GAP_SECONDS = 1.0
MAX_PLOT_CHANNELS = 12
CONTEXT_SECONDS = 4.0
METHOD_SAMPLES = 64        # common waveform length for DTW, DFA, and DWT
DTW_RADIUS = 8             # Sakoe–Chiba warping radius, in resampled points
OUTPUT_CSV = Path("/content/extreme_event_intervals.csv")

assert WINDOW_SECONDS > 0 and 0 < STEP_SECONDS <= WINDOW_SECONDS
assert 0 < CHANNEL_FRACTION <= 1 and THRESHOLD_MAD > 0
assert METHOD_SAMPLES >= 32 and 1 <= DTW_RADIUS < METHOD_SAMPLES

## 3. Upload and load the EDF

Upload `sEEG-HFOs-8.edf` when Colab prompts. The upload remains local to the temporary Colab runtime. The loader also accepts a file already placed in `/content` or `/content/dataset`.

In [ ]:
candidates = [
    Path("/content") / EDF_NAME,
    Path("/content/dataset") / EDF_NAME,
    Path.cwd() / EDF_NAME,
    Path.cwd() / "dataset" / EDF_NAME,
]
edf_path = next((path for path in candidates if path.is_file()), None)

if edf_path is None:
    try:
        from google.colab import files
    except ImportError as exc:
        searched = "\n".join(f"  - {path.resolve()}" for path in candidates)
        raise FileNotFoundError(f"{EDF_NAME} was not found. Searched:\n{searched}") from exc
    print(f"Select {EDF_NAME} in the upload dialog.")
    uploaded = files.upload()
    if EDF_NAME not in uploaded:
        raise FileNotFoundError(f"The uploaded file must be named {EDF_NAME!r}.")
    edf_path = Path("/content") / EDF_NAME

raw = mne.io.read_raw_edf(edf_path, preload=False, verbose="WARNING", encoding="cp1251")
marker_channels = [name for name in raw.ch_names if name.strip().upper().startswith("MKR")]
signal_channels = [name for name in raw.ch_names if name not in marker_channels]
if not signal_channels:
    raise ValueError("No non-MKR channels remain.")

sfreq = float(raw.info["sfreq"])
duration_s = raw.n_times / sfreq
print(f"Loaded: {edf_path}")
print(f"Duration: {duration_s:.3f} s | sampling rate: {sfreq:g} Hz")
print(f"Analyzing all {len(signal_channels)} non-MKR channels")
print(f"Excluded {len(marker_channels)} marker channel(s): {marker_channels or 'none'}")
display(pd.DataFrame({"analyzed_channel": signal_channels}))

## 3b. Locate the EDF's own annotated event (no apriori time)

The EDF+ annotation channel is metadata the clinician who scored the recording embedded directly in the file — read here the same way `meas_date` or the channel list is read, not typed in as a separate number. Every annotation is decoded with `cp1251` (Windows-1251 Cyrillic; MNE's default UTF-8 decode raises on it) and searched for text naming a seizure. Every other annotation within `ANNOTATION_CLUSTER_GAP_SECONDS` of a match is folded into the same event, so a clinician's separate notes about one seizure (e.g. an "onset?" query beside a "seizure" tag) do not read as unrelated events. If no such annotation exists, `known_peak_seconds` stays `None` and every figure below simply omits the marker — the rest of the notebook does not depend on it.


In [ ]:
SEIZURE_KEYWORDS = ("\u043f\u0440\u0438\u0441\u0442\u0443\u043f", "\u0441\u0443\u0434\u043e\u0440\u043e\u0433", "seizure", "ictal", "\u0431\u0442\u043a\u043f", "tcs")
ANNOTATION_CLUSTER_GAP_SECONDS = 10.0

annotation_onsets = [float(onset) for onset in raw.annotations.onset]
annotation_descriptions = [str(description) for description in raw.annotations.description]


def cluster_seizure_annotation(onsets, descriptions):
    """Mirrors extreme_event_agent.edf_workflow._cluster_seizure_annotation."""
    matches = {index for index, description in enumerate(descriptions)
              if any(keyword in description.lower() for keyword in SEIZURE_KEYWORDS)}
    if not matches:
        return None
    order = sorted(range(len(onsets)), key=lambda index: onsets[index])
    groups = [[order[0]]]
    for index in order[1:]:
        if onsets[index] - onsets[groups[-1][-1]] <= ANNOTATION_CLUSTER_GAP_SECONDS:
            groups[-1].append(index)
        else:
            groups.append([index])
    group = next(group for group in groups if matches & set(group))
    anchor = min((index for index in group if index in matches), key=lambda index: onsets[index])
    return {
        "time_s": onsets[anchor],
        "label": descriptions[anchor],
        "cluster": [(onsets[index], descriptions[index]) for index in group],
    }


known_peak = cluster_seizure_annotation(annotation_onsets, annotation_descriptions)
if known_peak is None:
    known_peak_seconds = None
    known_peak_label = None
    known_peak_duration_s = None
    print("No seizure-labelled annotation found in this EDF's own annotation channel; "
          "figures below will not carry a known-peak marker.")
else:
    known_peak_seconds = known_peak["time_s"]
    known_peak_label = known_peak["label"]
    cluster_start = known_peak["cluster"][0][0]
    cluster_end = known_peak["cluster"][-1][0]
    known_peak_duration_s = max(cluster_end - cluster_start, WINDOW_SECONDS)
    print(f"EDF-annotated peak: {known_peak_seconds:.3f} s ({known_peak_label!r})")
    print("Full matched annotation cluster (verbatim from the file):")
    for onset, description in known_peak["cluster"]:
        print(f"  {onset:.3f} s: {description!r}")


## 4. Extract common window data from the complete recording

The EDF remains disk-backed. Windows are read in batches to limit memory while avoiding one disk access per window. MNE returns volts, which are converted to microvolts. Non-finite samples are replaced by the channel median within that window and counted for quality review.

In addition to four conventional amplitude/transient features, each waveform is robustly centered, scaled, and linearly resampled to `METHOD_SAMPLES`. This compact representation makes the DTW, DFA, and DWT comparisons tractable while preserving every channel and every analysis window. Kuramoto phase synchrony is computed later from full-rate data, not from these resampled signals.

In [ ]:
window_samples = int(round(WINDOW_SECONDS * sfreq))
step_samples = int(round(STEP_SECONDS * sfreq))
if window_samples < 2 or step_samples < 1:
    raise ValueError("Window or step is too short for this sampling frequency.")
window_starts = np.arange(0, raw.n_times - window_samples + 1, step_samples, dtype=int)
if window_starts.size == 0:
    raise ValueError("The recording is shorter than one analysis window.")

feature_names = ["rms", "peak_to_peak", "line_length", "difference_rms"]
features = np.empty((len(window_starts), len(signal_channels), len(feature_names)), dtype=np.float32)
method_waveforms = np.empty((len(window_starts), len(signal_channels), METHOD_SAMPLES), dtype=np.float32)
nonfinite_fraction = np.empty((len(window_starts), len(signal_channels)), dtype=np.float32)
source_grid = np.linspace(0.0, 1.0, window_samples)
target_grid = np.linspace(0.0, 1.0, METHOD_SAMPLES)
batch_windows = 128

for batch_start in range(0, len(window_starts), batch_windows):
    batch_stop = min(batch_start + batch_windows, len(window_starts))
    first_sample = int(window_starts[batch_start])
    last_sample = int(window_starts[batch_stop - 1] + window_samples)
    block = raw.get_data(picks=signal_channels, start=first_sample, stop=last_sample) * 1e6
    for output_index in range(batch_start, batch_stop):
        relative_start = int(window_starts[output_index] - first_sample)
        values = block[:, relative_start:relative_start + window_samples].astype(float, copy=True)
        finite = np.isfinite(values)
        nonfinite_fraction[output_index] = 1.0 - finite.mean(axis=1)
        for channel_index in np.flatnonzero(~finite.all(axis=1)):
            good = finite[channel_index]
            replacement = np.median(values[channel_index, good]) if good.any() else 0.0
            values[channel_index, ~good] = replacement
        centered = values - np.median(values, axis=1, keepdims=True)
        differences = np.diff(values, axis=1)
        features[output_index, :, 0] = np.sqrt(np.mean(centered ** 2, axis=1))
        features[output_index, :, 1] = np.ptp(values, axis=1)
        features[output_index, :, 2] = np.mean(np.abs(differences), axis=1)
        features[output_index, :, 3] = np.sqrt(np.mean(differences ** 2, axis=1))
        scale = np.median(np.abs(centered), axis=1, keepdims=True) * 1.4826
        scale = np.where(scale > 1e-12, scale, np.std(centered, axis=1, keepdims=True))
        normalized = centered / np.where(scale > 1e-12, scale, 1.0)
        for channel_index in range(len(signal_channels)):
            method_waveforms[output_index, channel_index] = np.interp(
                target_grid, source_grid, normalized[channel_index]
            )
    if batch_start == 0 or batch_stop == len(window_starts) or batch_stop % 512 == 0:
        print(f"Processed {batch_stop:,}/{len(window_starts):,} windows")

window_start_s = window_starts / sfreq
window_end_s = np.minimum((window_starts + window_samples) / sfreq, duration_s)
print("Conventional features [window, channel, feature]:", features.shape)
print("Compact waveforms [window, channel, sample]:", method_waveforms.shape)
print(f"Maximum non-finite fraction in any channel/window: {nonfinite_fraction.max():.3%}")

## 5. Dynamic Time Warping (DTW; sometimes called “Domain Time Wrapping”)

**Short review.** DTW measures waveform dissimilarity while allowing local nonlinear stretching or compression of the time axis. Here each normalized channel/window is compared with that channel’s median recording template. A Sakoe–Chiba band limits the alignment path, reducing computation and preventing implausibly large time shifts. A high distance means the waveform shape is unusual for that contact. DTW can also respond strongly to artifacts, and its distance is not a physiological biomarker by itself.

In [ ]:
dtw_templates = np.median(method_waveforms, axis=0).astype(np.float32)

@njit(parallel=True)
def constrained_dtw_all(waveforms, templates, radius):
    n_windows, n_channels, length = waveforms.shape
    output = np.empty((n_windows, n_channels), dtype=np.float32)
    infinity = np.float32(1e30)
    for item in prange(n_windows * n_channels):
        window_index = item // n_channels
        channel_index = item % n_channels
        previous = np.full(length + 1, infinity, dtype=np.float32)
        current = np.full(length + 1, infinity, dtype=np.float32)
        previous[0] = 0.0
        for i in range(1, length + 1):
            current[:] = infinity
            lower = max(1, i - radius)
            upper = min(length, i + radius)
            for j in range(lower, upper + 1):
                difference = waveforms[window_index, channel_index, i - 1] - templates[channel_index, j - 1]
                cost = difference * difference
                best = min(previous[j], current[j - 1], previous[j - 1])
                current[j] = cost + best
            previous, current = current, previous
        output[window_index, channel_index] = np.sqrt(previous[length] / length)
    return output

dtw_distance = constrained_dtw_all(method_waveforms, dtw_templates, DTW_RADIUS)
print("DTW distances [window, channel]:", dtw_distance.shape)

## 6. Detrended Fluctuation Analysis (DFA)

**Short review.** DFA estimates scale-free temporal correlation while suppressing local polynomial trends. The centered signal is cumulatively integrated, split into boxes at several scales, linearly detrended within each box, and summarized by the slope of log fluctuation versus log box size. The slope (often called the DFA exponent) changes when temporal organization changes. With short event windows it is a comparative feature—not a reliable estimate of long-range scaling—and should not be over-interpreted physiologically.

In [ ]:
@njit(parallel=True)
def dfa_exponents(waveforms):
    n_windows, n_channels, length = waveforms.shape
    scales = np.array([4, 8, 16, 32])
    output = np.empty((n_windows, n_channels), dtype=np.float32)
    for item in prange(n_windows * n_channels):
        window_index = item // n_channels
        channel_index = item % n_channels
        integrated = np.cumsum(waveforms[window_index, channel_index])
        log_scales = np.empty(len(scales), dtype=np.float64)
        log_fluctuations = np.empty(len(scales), dtype=np.float64)
        valid = 0
        for scale in scales:
            boxes = length // scale
            if boxes < 2:
                continue
            squared_sum = 0.0
            point_count = 0
            x_mean = (scale - 1) / 2.0
            x_variance = 0.0
            for x in range(scale):
                x_variance += (x - x_mean) ** 2
            for box in range(boxes):
                offset = box * scale
                y_mean = 0.0
                for x in range(scale):
                    y_mean += integrated[offset + x]
                y_mean /= scale
                covariance = 0.0
                for x in range(scale):
                    covariance += (x - x_mean) * (integrated[offset + x] - y_mean)
                slope = covariance / x_variance
                intercept = y_mean - slope * x_mean
                for x in range(scale):
                    residual = integrated[offset + x] - (intercept + slope * x)
                    squared_sum += residual * residual
                    point_count += 1
            fluctuation = np.sqrt(squared_sum / point_count)
            if fluctuation > 1e-12:
                log_scales[valid] = np.log(scale)
                log_fluctuations[valid] = np.log(fluctuation)
                valid += 1
        if valid < 2:
            output[window_index, channel_index] = np.nan
        else:
            x_mean = np.mean(log_scales[:valid])
            y_mean = np.mean(log_fluctuations[:valid])
            numerator = np.sum((log_scales[:valid] - x_mean) * (log_fluctuations[:valid] - y_mean))
            denominator = np.sum((log_scales[:valid] - x_mean) ** 2)
            output[window_index, channel_index] = numerator / denominator
    return output

dfa_alpha = dfa_exponents(method_waveforms)
print("DFA exponents [window, channel]:", dfa_alpha.shape)

## 7. Discrete Wavelet Transform (DWT)

**Short review.** The DWT separates each window into an approximation and progressively finer detail components. Squared coefficient magnitude represents energy at each dyadic scale, allowing brief sharp activity and slower changes to contribute separately. This notebook uses a `db4` wavelet and summarizes log energy across its detail bands. The bands are approximate and depend on sampling/resampling; padding and window boundaries can create edge effects.

In [ ]:
DWT_WAVELET = "db4"
DWT_LEVEL = min(4, pywt.dwt_max_level(METHOD_SAMPLES, pywt.Wavelet(DWT_WAVELET).dec_len))
dwt_coefficients = pywt.wavedec(method_waveforms, DWT_WAVELET, level=DWT_LEVEL,
                                mode="symmetric", axis=-1)
# wavedec returns [approximation, detail_level, ..., detail_1].
dwt_log_detail_energy = np.stack(
    [np.log1p(np.mean(np.square(detail), axis=-1)) for detail in dwt_coefficients[1:]],
    axis=-1,
).astype(np.float32)
print("DWT log-detail energies [window, channel, scale]:", dwt_log_detail_energy.shape)

## 8. Kuramoto phase synchronization in canonical brain rhythms

**Short review.** The Kuramoto order parameter measures instantaneous phase alignment across oscillators: zero indicates dispersed phases and one indicates perfect alignment. Here the “oscillators” are all retained contacts. Full-rate signals are band-pass filtered into delta (0.5–4 Hz), theta (4–8 Hz), alpha (8–13 Hz), beta (13–30 Hz), and gamma (30–80 Hz); Hilbert phases are then combined across channels. The window metric is mean phase synchrony. Volume conduction, the common reference, filtering, and edge effects can inflate synchrony, so this is an exploratory network feature rather than proof of coupling.

In [ ]:
rhythm_bands = {
    "delta": (0.5, 4.0),
    "theta": (4.0, 8.0),
    "alpha": (8.0, 13.0),
    "beta": (13.0, 30.0),
    "gamma": (30.0, 80.0),
}
valid_bands = {name: (low, high) for name, (low, high) in rhythm_bands.items()
               if high < sfreq / 2}
if not valid_bands:
    raise ValueError("Sampling frequency is too low for the configured rhythm bands.")
kuramoto_order = np.empty((len(window_starts), len(valid_bands)), dtype=np.float32)

# Each expanded batch includes padding, which is discarded before window summaries.
batch_windows = 128
filter_margin = int(round(max(4.0, WINDOW_SECONDS) * sfreq))
for batch_start in range(0, len(window_starts), batch_windows):
    batch_stop = min(batch_start + batch_windows, len(window_starts))
    core_first = int(window_starts[batch_start])
    core_last = int(window_starts[batch_stop - 1] + window_samples)
    read_first = max(0, core_first - filter_margin)
    read_last = min(raw.n_times, core_last + filter_margin)
    block = raw.get_data(picks=signal_channels, start=read_first, stop=read_last)
    block = np.nan_to_num(block - np.nanmedian(block, axis=1, keepdims=True))
    for band_index, (band_name, (low_hz, high_hz)) in enumerate(valid_bands.items()):
        sos = butter(4, [low_hz, high_hz], btype="bandpass", fs=sfreq, output="sos")
        phase = np.angle(hilbert(sosfiltfilt(sos, block, axis=1), axis=1))
        instantaneous_order = np.abs(np.mean(np.exp(1j * phase), axis=0))
        for output_index in range(batch_start, batch_stop):
            relative_start = int(window_starts[output_index] - read_first)
            kuramoto_order[output_index, band_index] = np.mean(
                instantaneous_order[relative_start:relative_start + window_samples]
            )
    print(f"Kuramoto: processed {batch_stop:,}/{len(window_starts):,} windows")

print("Kuramoto order [window, rhythm]:", kuramoto_order.shape)
display(pd.DataFrame(kuramoto_order, columns=valid_bands).describe())

## 9. Combine methods and form event intervals

Each method is normalized against its own recording-wide median/MAD distribution before combination. Conventional, DTW, DFA, and DWT channel features are reduced using the most abnormal fraction of contacts; Kuramoto uses the most abnormal rhythm bands. DFA and Kuramoto are scored two-sided because either an increase or decrease can be unusual. The final ensemble is the mean of the three strongest method scores per window, so one weak method does not conceal agreement among the others.

Adjacent threshold crossings are merged. `start_s` and `end_s` cover the union of their overlapping windows. If nothing crosses the automatic ensemble threshold, the recording maximum is still returned with `detected=False`.

If Section 3b located an EDF-annotated peak, this section also answers a second, separate question: not "what is the single strongest window in the recording", but "how would *each* method, alone, have scored *that specific, known* moment, relative to the rest of the recording"? That is the math answer to "how could an extreme event be predicted" for every method individually, not just the combined ensemble.

In [ ]:
def robust_z(values, axis=0, two_sided=False):
    center = np.nanmedian(values, axis=axis, keepdims=True)
    mad = np.nanmedian(np.abs(values - center), axis=axis, keepdims=True)
    fallback = np.nanstd(values, axis=axis, keepdims=True)
    scale = np.where(1.4826 * mad > 1e-12, 1.4826 * mad,
                     np.where(fallback > 1e-12, fallback, 1.0))
    z = (values - center) / scale
    z = np.abs(z) if two_sided else np.maximum(z, 0.0)
    return np.clip(np.nan_to_num(z, nan=0.0, posinf=25.0, neginf=0.0), 0.0, 25.0)

n_top_channels = max(1, int(np.ceil(CHANNEL_FRACTION * len(signal_channels))))
def channel_consensus(channel_values):
    return np.sort(channel_values, axis=1)[:, -n_top_channels:].mean(axis=1)

basic_channel_score = np.sort(robust_z(features, axis=0), axis=2)[:, :, -2:].mean(axis=2)
dtw_channel_score = robust_z(dtw_distance, axis=0)
dfa_channel_score = robust_z(dfa_alpha, axis=0, two_sided=True)
dwt_channel_score = np.max(robust_z(dwt_log_detail_energy, axis=0), axis=2)
kuramoto_band_score = robust_z(kuramoto_order, axis=0, two_sided=True)

method_scores = pd.DataFrame({
    "time_domain": channel_consensus(basic_channel_score),
    "dtw": channel_consensus(dtw_channel_score),
    "dfa": channel_consensus(dfa_channel_score),
    "dwt": channel_consensus(dwt_channel_score),
    "kuramoto": np.max(kuramoto_band_score, axis=1),
})
method_score_z = robust_z(method_scores.to_numpy(), axis=0)
global_score = np.sort(method_score_z, axis=1)[:, -3:].mean(axis=1)
score_center = float(np.median(global_score))
score_scale = float(max(1.4826 * np.median(np.abs(global_score - score_center)),
                        np.std(global_score), 1e-12))
automatic_threshold = score_center + THRESHOLD_MAD * score_scale
flagged = global_score >= automatic_threshold

# Combined per-channel score is used only to identify contacts for review.
channel_score = np.mean(np.stack([
    basic_channel_score, dtw_channel_score, dfa_channel_score, dwt_channel_score
], axis=2), axis=2)

def merged_runs(mask, max_gap_windows):
    indices = np.flatnonzero(mask)
    if not indices.size:
        return []
    groups, current = [], [int(indices[0])]
    for index in indices[1:]:
        if int(index) - current[-1] <= max_gap_windows + 1:
            current.append(int(index))
        else:
            groups.append(current)
            current = [int(index)]
    groups.append(current)
    return groups

runs = merged_runs(flagged, int(np.floor(MERGE_GAP_SECONDS / STEP_SECONDS)))
detected = bool(runs)
if not runs:
    runs = [[int(np.argmax(global_score))]]
rows = []
for event_id, run in enumerate(runs, start=1):
    covered = np.arange(run[0], run[-1] + 1)
    peak_index = int(covered[np.argmax(global_score[covered])])
    strongest = np.argsort(channel_score[peak_index])[::-1][:n_top_channels]
    dominant_methods = method_scores.columns[np.argsort(method_score_z[peak_index])[::-1][:3]]
    rows.append({
        "event_id": event_id, "detected": detected,
        "start_s": float(window_start_s[run[0]]),
        "end_s": float(window_end_s[run[-1]]),
        "duration_s": float(window_end_s[run[-1]] - window_start_s[run[0]]),
        "peak_time_s": float(window_start_s[peak_index]),
        "peak_score": float(global_score[peak_index]),
        "dominant_methods": ", ".join(dominant_methods),
        "top_channels": ", ".join(signal_channels[index] for index in strongest),
        "peak_window_index": peak_index,
    })
events = pd.DataFrame(rows).sort_values("peak_score", ascending=False).reset_index(drop=True)
events.insert(0, "rank", np.arange(1, len(events) + 1))
events.to_csv(OUTPUT_CSV, index=False)
print(f"Automatic ensemble threshold: {automatic_threshold:.3f}")
print(f"Detected {len(events)} event interval(s)." if detected else
      "No crossing; returning the recording maximum with detected=False.")
display(events.drop(columns="peak_window_index"))
print(f"Saved table: {OUTPUT_CSV}")

if known_peak_seconds is not None:
    known_peak_window = int(np.argmin(np.abs(window_start_s - known_peak_seconds)))
    method_names = list(method_scores.columns)
    percentiles = [100.0 * float((method_score_z[:, index] <= method_score_z[known_peak_window, index]).mean())
                  for index in range(len(method_names))]
    ensemble_percentile = 100.0 * float((global_score <= global_score[known_peak_window]).mean())
    ensemble_rank = int((global_score > global_score[known_peak_window]).sum()) + 1
    known_peak_report = pd.DataFrame({
        "method": method_names + ["ensemble (combined)"],
        "score_at_known_peak": [float(method_score_z[known_peak_window, index])
                               for index in range(len(method_names))] + [float(global_score[known_peak_window])],
        "percentile_in_recording": percentiles + [ensemble_percentile],
    }).sort_values("percentile_in_recording", ascending=False).reset_index(drop=True)
    print()
    print(f"How would each method alone have scored the EDF-annotated peak at "
          f"{known_peak_seconds:.3f} s ({known_peak_label!r})?")
    print(f"Nearest analysis window starts at {window_start_s[known_peak_window]:.3f} s "
          f"(recording-wide ensemble rank {ensemble_rank} of {len(global_score)}; "
          f"{'ABOVE' if global_score[known_peak_window] >= automatic_threshold else 'below'} "
          f"the automatic threshold of {automatic_threshold:.3f}).")
    display(known_peak_report)
else:
    known_peak_window = None


## 10. Visualize the ensemble and method scores across the recording

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(17, 9), sharex=True, constrained_layout=True)
axes[0].plot(window_start_s, global_score, color="navy", linewidth=0.8, label="Ensemble score")
axes[0].axhline(automatic_threshold, color="crimson", linestyle="--",
                label=f"Automatic threshold ({automatic_threshold:.2f})")
for _, event in events.iterrows():
    color = "crimson" if event["detected"] else "darkorange"
    axes[0].axvspan(event["start_s"], event["end_s"], color=color, alpha=0.18)
axes[0].set(ylabel="Robust ensemble score", title="Whole-recording extreme-event scan")
axes[0].legend(loc="upper right")
for method in method_scores.columns:
    axes[1].plot(window_start_s, method_score_z[:, method_scores.columns.get_loc(method)],
                 linewidth=0.7, label=method)
axes[1].set(xlabel="Time (s)", ylabel="Robust method score",
            title="Contributions from each analysis method")
axes[1].legend(ncol=5, loc="upper right")
if known_peak_seconds is not None:
    for axis in axes:
        axis.axvline(known_peak_seconds, color="teal", linewidth=1.5, zorder=5)
    axes[0].annotate(f"EDF-annotated peak\n{known_peak_label}\n{known_peak_seconds:.3f} s",
                     (known_peak_seconds, axes[0].get_ylim()[1]), xytext=(6, -6),
                     textcoords="offset points", va="top", color="teal", fontsize=8, fontweight="bold")
for axis in axes:
    axis.margins(x=0)
plt.show()

## 11. Visualize the strongest event and its most involved channels

The first panel shows raw microvolt traces, individually centered and divided by their recording-level RMS baseline before vertical offsets. This preserves waveform shape while preventing one large-amplitude contact from hiding the others. The heatmap shows each channel’s anomaly score around the event; all channels were included in detection even though only the strongest are displayed.

In [ ]:
def visualize_event_window(peak_index, span_start_s, span_end_s, title, marker_color, marker_label):
    """Plot normalized traces and a per-channel anomaly heatmap around one window.

    ``peak_index`` selects which window's ``channel_score`` ranks the channels
    shown. ``span_start_s``/``span_end_s`` are shaded on the trace plot; the
    surrounding ``CONTEXT_SECONDS`` is added for temporal context. If Section 3b
    located an EDF-annotated peak, it is marked (teal) on both panels whenever
    it falls within the plotted window, regardless of which event this call is
    centred on, so the two are always directly comparable on the same figure.
    """
    plot_count = min(MAX_PLOT_CHANNELS, len(signal_channels))
    plot_indices = np.argsort(channel_score[peak_index])[::-1][:plot_count]
    plot_channels = [signal_channels[index] for index in plot_indices]

    plot_start_s = max(0.0, span_start_s - CONTEXT_SECONDS)
    plot_end_s = min(duration_s, span_end_s + CONTEXT_SECONDS)
    start_sample = int(np.floor(plot_start_s * sfreq))
    stop_sample = min(raw.n_times, int(np.ceil(plot_end_s * sfreq)))
    trace = raw.get_data(picks=plot_channels, start=start_sample, stop=stop_sample) * 1e6
    trace_times = np.arange(start_sample, stop_sample) / sfreq
    trace -= np.nanmedian(trace, axis=1, keepdims=True)
    baseline_rms = np.median(features[:, plot_indices, 0], axis=0)
    baseline_rms = np.where(baseline_rms > 1e-12, baseline_rms, 1.0)
    normalized_trace = trace / baseline_rms[:, None]
    offsets = np.arange(plot_count)[::-1] * 8.0

    show_known_peak = (known_peak_seconds is not None
                       and plot_start_s <= known_peak_seconds <= plot_end_s)

    fig, ax = plt.subplots(figsize=(17, max(6, 0.55 * plot_count)))
    for values, offset, name in zip(normalized_trace, offsets, plot_channels):
        ax.plot(trace_times, values + offset, linewidth=0.65, color="black")
    ax.axvspan(span_start_s, span_end_s, color=marker_color, alpha=0.18, label=marker_label)
    if show_known_peak:
        ax.axvline(known_peak_seconds, color="teal", linewidth=1.5, zorder=5,
                   label=f"EDF-annotated peak ({known_peak_label})")
    ax.set_yticks(offsets, labels=plot_channels)
    ax.set(xlabel="Time (s)", ylabel="Channel (normalized traces, vertically offset)", title=title)
    ax.legend(loc="upper right")
    ax.margins(x=0)
    plt.tight_layout()
    plt.show()

    nearby = (window_start_s >= plot_start_s) & (window_start_s <= plot_end_s)
    plt.figure(figsize=(17, max(5, 0.45 * plot_count)))
    heatmap_axis = sns.heatmap(channel_score[nearby][:, plot_indices].T, cmap="magma",
                               xticklabels=False, yticklabels=plot_channels,
                               cbar_kws={"label": "Channel anomaly score"})
    if show_known_peak:
        nearby_times = window_start_s[nearby]
        known_peak_column = int(np.argmin(np.abs(nearby_times - known_peak_seconds)))
        heatmap_axis.axvline(known_peak_column, color="teal", linewidth=2)
        heatmap_axis.text(known_peak_column, -0.5, "EDF-annotated peak", color="teal",
                          ha="center", va="bottom", fontsize=8, fontweight="bold")
    plt.xlabel(f"Windows from {plot_start_s:.3f} to {plot_end_s:.3f} s")
    plt.ylabel("Strongest channels at this window")
    plt.title(title)
    plt.tight_layout()
    plt.show()
    return plot_channels


strongest_event = events.iloc[0]
visualize_event_window(
    peak_index=int(strongest_event["peak_window_index"]),
    span_start_s=float(strongest_event["start_s"]), span_end_s=float(strongest_event["end_s"]),
    title=f"Strongest event: {strongest_event['start_s']:.3f}\u2013{strongest_event['end_s']:.3f} s",
    marker_color="crimson", marker_label="Returned event interval")

print("Returned strongest-event time period:")
print(f"  {strongest_event['start_s']:.6f} s to {strongest_event['end_s']:.6f} s")
print(f"Peak window begins at {strongest_event['peak_time_s']:.6f} s")
print(f"Most involved channels: {strongest_event['top_channels']}")

## 11b. Visualize the EDF-annotated peak

The same trace-and-heatmap view as above, but centred on the EDF's own annotated seizure marker (Section 3b) instead of whichever window the blind ensemble ranked highest. The strongest-event view above already marks this same time in teal when it falls in range, so this section exists for the reverse case: when the two disagree, this shows what the recording actually looks like at the real event, with its own top-ranked channels — not the channels selected for the (possibly unrelated) strongest window.


In [ ]:
if known_peak_seconds is not None:
    known_peak_span_end = known_peak_seconds + (known_peak_duration_s or WINDOW_SECONDS)
    visualize_event_window(
        peak_index=known_peak_window,
        span_start_s=known_peak_seconds, span_end_s=known_peak_span_end,
        title=f"EDF-annotated peak: {known_peak_label} at {known_peak_seconds:.3f} s",
        marker_color="teal", marker_label="EDF-annotated event span")
    print("EDF-annotated peak time period:")
    print(f"  {known_peak_seconds:.6f} s ({known_peak_label!r})")
    print(f"Nearest analysis window: {window_start_s[known_peak_window]:.6f} s "
          f"(ensemble score {global_score[known_peak_window]:.3f}, "
          f"{'above' if global_score[known_peak_window] >= automatic_threshold else 'below'} threshold)")
else:
    print("No EDF-annotated peak found in this file; nothing to compare against here.")

## 12. Interpretation and tuning

1. Inspect every ranked interval in `events`, not only the strongest plot.
2. A single noisy contact can still create a focal score. Inspect its raw trace and compare neighboring contacts.
3. To demand broader spatial agreement, increase `CHANNEL_FRACTION` (for example, to `0.25`).
4. To find more candidates, reduce `THRESHOLD_MAD`; to increase specificity, raise it. Re-run from feature scoring after changing those two values.
5. Shorter windows localize brief spikes but are more variable; longer windows favor sustained changes.
6. The returned bounds have window-level precision and include the full overlapping analysis windows. They are not onset/offset annotations at sample precision.
7. Validate candidate events with a qualified clinician and with the unprocessed EDF, montage/reference, video, and other clinical context.